<a href="https://colab.research.google.com/github/mkobycheva/recommendation-system/blob/svd/notebooks/01_1_svd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!test -d /content/recommendation-system/.git \
  || git clone -b als https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin main
!git checkout main
!git reset --hard origin/main

# Create and switch to the new agent_als branch
!git checkout -b svd

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

/content/recommendation-system
From https://github.com/mkobycheva/recommendation-system
 * branch            main       -> FETCH_HEAD
Switched to branch 'main'
Your branch is up to date with 'origin/main'.
HEAD is now at 6fef0c2 Merge pull request #4 from mkobycheva/als
fatal: A branch named 'svd' already exists.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os

import numpy as np
import pandas as pd
import numpy as np
from scipy.sparse.linalg import svds

from src.data.build_matrix import add_domain_item_ids, build_explicit_svd_matrix
from src.evaluation.metrics import map_at_k

## Data preparation

In [8]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [9]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

,user_id,item_id,domain
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0061713244,books
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0871139634,books
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0385521685,books
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0848732634,books
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0151014981,books


In [10]:
interaction_matrix = build_explicit_svd_matrix(train_df)

user_item_train = interaction_matrix.user_item_train
user2idx = interaction_matrix.user2idx
item2idx = interaction_matrix.item2idx
idx2item = interaction_matrix.idx2item
item_domain = interaction_matrix.item_domain
train_seen_idx_by_user = interaction_matrix.train_seen_idx_by_user
domain_item_indices = interaction_matrix.domain_item_indices

n_users, n_items = user_item_train.shape

print(f'users={n_users:,}, items={n_items:,}, train_interactions={user_item_train.nnz:,}')

users=127,188, items=587,064, train_interactions=3,750,813


## Decomposing the matrix

In [11]:
# Subtract each user's mean rating from their entries
row_means = np.array(user_item_train.sum(axis=1)).flatten() / \
            np.diff(user_item_train.indptr).clip(min=1)

centered = user_item_train.copy().astype(np.float32)
for i in range(centered.shape[0]):
    start, end = centered.indptr[i], centered.indptr[i+1]
    centered.data[start:end] -= row_means[i]

N_FACTORS = 100

U, sigma, Vt = svds(centered, k=N_FACTORS)

U     = U[:, ::-1]    # (127188, 100)
sigma = sigma[::-1]   # (100,)
Vt    = Vt[::-1, :]   # (100, 587064)

In [12]:
user_factors = U @ np.diag(sigma)   # (127188, 100)
item_factors = Vt.T                 # (587064, 100)

In [13]:
idx2item_array = np.array([idx2item[i] for i in range(len(idx2item))])

## Model evaluation


In [14]:
def get_domain_recommendations_batched(domain, relevant_by_user, K=10, batch_size=512):
    """Generate top-K recommendations restricted to one domain, in batches."""
    domain_indices = domain_item_indices[domain]            # global item indices for this domain
    domain_item_factors = item_factors[domain_indices]      # (n_domain_items, n_factors)
    domain_idx2item = idx2item_array[domain_indices]        # item_id strings, aligned with domain_indices

    user_ids = list(relevant_by_user.keys())
    recommended = {}

    for batch_start in range(0, len(user_ids), batch_size):
        batch_user_ids = user_ids[batch_start : batch_start + batch_size]
        batch_user_idx = [user2idx[uid] for uid in batch_user_ids]

        # Score every domain item for the whole batch of users at once
        batch_scores = user_factors[batch_user_idx] @ domain_item_factors.T  # (batch_size, n_domain_items)

        for i, user_id in enumerate(batch_user_ids):
            scores = batch_scores[i].copy()  # .copy() so masking doesn't corrupt the shared batch array

            # Mask out items this user has already seen, within this domain
            seen_global = train_seen_idx_by_user[user2idx[user_id]]
            seen_mask = np.isin(domain_indices, list(seen_global))
            scores[seen_mask] = -np.inf

            # Top K within domain
            k = min(K, len(scores))
            top_k_local = np.argpartition(scores, -k)[-k:]
            top_k_local = top_k_local[np.argsort(scores[top_k_local])[::-1]]

            recommended[user_id] = list(domain_idx2item[top_k_local])

    return recommended

In [15]:
books_relevant_valid = (
    valid_df[valid_df["domain"] == "books"]
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)
movies_relevant_valid = (
    valid_df[valid_df["domain"] == "movies"]
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

books_recommended_valid = get_domain_recommendations_batched("books", books_relevant_valid, K=10)
movies_recommended_valid = get_domain_recommendations_batched("movies", movies_relevant_valid, K=10)

In [16]:
K = 10

books_valid_map = map_at_k(books_recommended_valid, books_relevant_valid, k=K)
movies_valid_map = map_at_k(movies_recommended_valid, movies_relevant_valid, k=K)
valid_macro_map = (books_valid_map + movies_valid_map) / 2

print(f"books_valid_map@{K}: {books_valid_map:.4f}")
print(f"movies_valid_map@{K}: {movies_valid_map:.4f}")
print(f"valid_macro_map@{K}: {valid_macro_map:.4f}")

books_valid_map@10: 0.0009
movies_valid_map@10: 0.0039
valid_macro_map@10: 0.0024


In [17]:
books_relevant_test = (
    test_df[test_df["domain"] == "books"]
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)
movies_relevant_test = (
    test_df[test_df["domain"] == "movies"]
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

books_recommended_test = get_domain_recommendations_batched("books", books_relevant_test, K=10)
movies_recommended_test = get_domain_recommendations_batched("movies", movies_relevant_test, K=10)

books_test_map = map_at_k(books_recommended_test, books_relevant_test, k=K)
movies_test_map = map_at_k(movies_recommended_test, movies_relevant_test, k=K)
test_macro_map = (books_test_map + movies_test_map) / 2

print(f"books_test_map@{K}: {books_test_map:.4f}")
print(f"movies_test_map@{K}: {movies_test_map:.4f}")
print(f"test_macro_map@{K}: {test_macro_map:.4f}")

books_test_map@10: 0.0005
movies_test_map@10: 0.0025
test_macro_map@10: 0.0015
